In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import joblib
import optuna
from optuna.pruners import MedianPruner
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, auc
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Helper Classes

In [ ]:
class LOFModel:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_train = None
        self.df_valid = None
        self.df_test = None
        self.list_feature_cols = None
        self.model = None
        self.arr_train_scores = None
        self.arr_valid_scores = None
        self.arr_test_scores = None
        self.flt_optimal_threshold = None
        self.dict_best_params = None

    def import_data(self):
        """Load preprocessed data from S3."""
        print("Loading preprocessed data from S3...")
        str_train_uri = f's3://{self.str_bucket}/02_preprocessing/train_processed.parquet'
        str_valid_uri = f's3://{self.str_bucket}/02_preprocessing/valid_processed.parquet'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_processed.parquet'
        
        self.df_train = pd.read_parquet(str_train_uri)
        self.df_valid = pd.read_parquet(str_valid_uri)
        self.df_test = pd.read_parquet(str_test_uri)
        
        self.list_feature_cols = [col for col in self.df_train.columns if col != 'isFraud']
        
        print(f"Training set: {self.df_train.shape}")
        print(f"Validation set: {self.df_valid.shape}")
        print(f"Test set: {self.df_test.shape}")
        print(f"Features: {len(self.list_feature_cols)}")
        
        return self.df_train, self.df_valid, self.df_test

    def tune_model(self, int_n_trials=20):
        """Tune hyperparameters using Optuna on validation set."""
        if self.df_train is None:
            raise ValueError("Data not loaded.")
        
        print(f"\nTuning Local Outlier Factor with {int_n_trials} trials (using validation set)...")
        
        def objective(trial):
            int_n_neighbors = trial.suggest_int('n_neighbors', 5, 50)
            flt_contamination = trial.suggest_float('contamination', 0.001, 0.1)
            
            model_temp = LocalOutlierFactor(
                n_neighbors=int_n_neighbors,
                contamination=flt_contamination,
                novelty=True,
                n_jobs=-1
            )
            
            model_temp.fit(self.df_train[self.list_feature_cols])
            arr_scores_valid = model_temp.score_samples(self.df_valid[self.list_feature_cols])
            
            # Use PR-AUC on validation set for objective
            arr_precision, arr_recall, _ = precision_recall_curve(self.df_valid['isFraud'].values, -arr_scores_valid)
            flt_pr_auc = auc(arr_recall, arr_precision)
            return flt_pr_auc
        
        study = optuna.create_study(direction='maximize', pruner=MedianPruner())
        study.optimize(objective, n_trials=int_n_trials, show_progress_bar=True)
        
        self.dict_best_params = study.best_params
        print(f"\nBest parameters found:")
        for str_key, val in self.dict_best_params.items():
            print(f"  {str_key}: {val}")
        print(f"Best Validation PR-AUC: {study.best_value:.4f}")
        
        return self.dict_best_params

    def fit_model(self):
        """Fit the LOF model with best parameters."""
        if self.df_train is None:
            raise ValueError("Data not loaded.")
        if self.dict_best_params is None:
            raise ValueError("Model not tuned. Call tune_model() first.")
        
        print("\nFitting Local Outlier Factor model...")
        
        self.model = LocalOutlierFactor(
            n_neighbors=self.dict_best_params['n_neighbors'],
            contamination=self.dict_best_params['contamination'],
            novelty=True,
            n_jobs=-1
        )
        
        self.model.fit(self.df_train[self.list_feature_cols])
        print("Model fitted successfully")
        
        return self.model

    def predict(self):
        """Generate anomaly scores for train, valid, and test sets."""
        if self.model is None:
            raise ValueError("Model not fitted.")
        
        print("\nGenerating anomaly scores...")
        
        self.arr_train_scores = self.model.score_samples(self.df_train[self.list_feature_cols])
        self.arr_valid_scores = self.model.score_samples(self.df_valid[self.list_feature_cols])
        self.arr_test_scores = self.model.score_samples(self.df_test[self.list_feature_cols])
        
        print(f"Train scores - Mean: {self.arr_train_scores.mean():.4f}, Std: {self.arr_train_scores.std():.4f}")
        print(f"Valid scores - Mean: {self.arr_valid_scores.mean():.4f}, Std: {self.arr_valid_scores.std():.4f}")
        print(f"Test scores - Mean: {self.arr_test_scores.mean():.4f}, Std: {self.arr_test_scores.std():.4f}")
        
        return self.arr_train_scores, self.arr_valid_scores, self.arr_test_scores

    def evaluate(self):
        """Evaluate model on test set using threshold selected on validation set."""
        if self.arr_test_scores is None or self.arr_valid_scores is None:
            raise ValueError("Predictions not generated.")
        
        print("\nEvaluating model...")
        
        # Find optimal threshold on validation set
        arr_y_valid = self.df_valid['isFraud'].values
        arr_valid_scores_neg = -self.arr_valid_scores
        
        arr_precision_v, arr_recall_v, arr_thresholds_v = precision_recall_curve(arr_y_valid, arr_valid_scores_neg)
        
        # Compute F1 from precision/recall arrays (vectorized, no loop)
        arr_f1_v = 2 * (arr_precision_v[:-1] * arr_recall_v[:-1]) / (arr_precision_v[:-1] + arr_recall_v[:-1] + 1e-10)
        int_best_idx = np.argmax(arr_f1_v)
        self.flt_optimal_threshold = arr_thresholds_v[int_best_idx]
        print(f"Optimal Threshold (from validation set): {self.flt_optimal_threshold:.4f}")
        
        # Evaluate on test set using that threshold
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        flt_roc_auc = roc_auc_score(arr_y_test, arr_scores_neg)
        print(f"Test ROC-AUC: {flt_roc_auc:.4f}")
        
        arr_precision, arr_recall, arr_thresholds = precision_recall_curve(arr_y_test, arr_scores_neg)
        flt_pr_auc = auc(arr_recall, arr_precision)
        print(f"Test PR-AUC: {flt_pr_auc:.4f}")
        
        arr_preds_optimal = (arr_scores_neg >= self.flt_optimal_threshold).astype(int)
        
        flt_precision = precision_score(arr_y_test, arr_preds_optimal)
        flt_recall = recall_score(arr_y_test, arr_preds_optimal)
        flt_f1 = f1_score(arr_y_test, arr_preds_optimal)
        
        print(f"Test Precision: {flt_precision:.4f}")
        print(f"Test Recall: {flt_recall:.4f}")
        print(f"Test F1-Score: {flt_f1:.4f}")
        
        return {
            'roc_auc': flt_roc_auc,
            'pr_auc': flt_pr_auc,
            'precision': flt_precision,
            'recall': flt_recall,
            'f1': flt_f1,
            'threshold': self.flt_optimal_threshold
        }

    def plot_anomaly_scores(self):
        """Plot distribution of anomaly scores."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        ax.hist(-self.arr_test_scores[arr_y_test == 0], bins=100, alpha=0.6, label='Non-Fraud', color='#2ecc71')
        ax.hist(-self.arr_test_scores[arr_y_test == 1], bins=100, alpha=0.6, label='Fraud', color='#e74c3c')
        
        ax.axvline(self.flt_optimal_threshold, color='black', linestyle='--', linewidth=2, label=f'Optimal Threshold: {self.flt_optimal_threshold:.4f}')
        
        ax.set_xlabel('Anomaly Score', fontsize=12, fontweight='bold')
        ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax.set_title('Local Outlier Factor Anomaly Score Distribution', fontsize=14, fontweight='bold')
        ax.legend()
        ax.set_yscale('log')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_anomaly_scores.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 01_anomaly_scores.png")

    def plot_precision_recall_curve(self):
        """Plot precision-recall curve."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        arr_precision, arr_recall, arr_thresholds = precision_recall_curve(arr_y_test, arr_scores_neg)
        flt_pr_auc = auc(arr_recall, arr_precision)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_recall, arr_precision, linewidth=2.5, label=f'PR Curve (AUC={flt_pr_auc:.4f})', color='coral')
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
        
        ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
        ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
        ax.set_title('Precision-Recall Curve - Local Outlier Factor', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_precision_recall_curve.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 02_precision_recall_curve.png")

    def plot_roc_curve(self):
        """Plot ROC curve."""
        if self.arr_test_scores is None:
            raise ValueError("Predictions not generated.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        
        flt_roc_auc = roc_auc_score(arr_y_test, arr_scores_neg)
        arr_fpr, arr_tpr, arr_thresholds = roc_curve(arr_y_test, arr_scores_neg)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_fpr, arr_tpr, linewidth=2.5, label=f'ROC Curve (AUC={flt_roc_auc:.4f})', color='coral')
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5, label='Random Classifier')
        
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curve - Local Outlier Factor', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_roc_curve.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 03_roc_curve.png")

    def plot_confusion_matrix(self):
        """Plot confusion matrix at optimal threshold."""
        if self.arr_test_scores is None or self.flt_optimal_threshold is None:
            raise ValueError("Predictions not generated or threshold not set.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_scores_neg = -self.arr_test_scores
        arr_preds = (arr_scores_neg >= self.flt_optimal_threshold).astype(int)
        
        cm = confusion_matrix(arr_y_test, arr_preds)
        
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False, ax=ax,
                    xticklabels=['Non-Fraud', 'Fraud'],
                    yticklabels=['Non-Fraud', 'Fraud'])
        
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_title('Confusion Matrix - Local Outlier Factor', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_confusion_matrix.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 04_confusion_matrix.png")

    def save_model(self):
        """Save trained model to disk and S3."""
        if self.model is None:
            raise ValueError("Model not fitted.")
        
        print("\nSaving model...")
        
        from io import BytesIO
        buffer = BytesIO()
        joblib.dump(self.model, buffer)
        buffer.seek(0)
        
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key='04_lof/lof_model.pkl',
                Body=buffer.getvalue()
            )
            print("Model saved to S3")
        except Exception as e:
            print(f"Error uploading to S3: {e}")

## Constants

In [ ]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'lof'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

## Load Data

In [ ]:
model_lof = LOFModel(str_bucket, str_dirname_output)
df_train, df_valid, df_test = model_lof.import_data()

## Hyperparameter Tuning

In [ ]:
dict_best_params = model_lof.tune_model(int_n_trials=20)

## Fit Model

In [ ]:
model = model_lof.fit_model()

## Generate Predictions

In [ ]:
arr_train_scores, arr_valid_scores, arr_test_scores = model_lof.predict()

## Model Evaluation

In [ ]:
dict_metrics = model_lof.evaluate()

## Generate Visualizations

In [ ]:
model_lof.plot_anomaly_scores()

In [ ]:
model_lof.plot_precision_recall_curve()

In [ ]:
model_lof.plot_roc_curve()

In [ ]:
model_lof.plot_confusion_matrix()

## Save Model

In [ ]:
model_lof.save_model()

## Completion

In [ ]:
print("\n=== LOCAL OUTLIER FACTOR MODELING COMPLETE ===")
print(f"\nMetrics Summary:")
for str_metric, val in dict_metrics.items():
    print(f"  {str_metric}: {val:.4f}")